# 3.3 — Manage, Monitor, and Optimize Costs

**Exam domain:** Gen AI Governance · **Weight:** 28%

## The problem this solves

The finance business partner forwards a bill that is three times last month's and asks one question:
what changed? Nobody on the data team can answer, because the nightly classification job, the search
service that nobody queries any more, the chatbot and the document pipeline all bill differently and
none of them appear in the warehouse credit report anyone is watching.

Cortex spend is not one number. Each service has its own billing unit and its own usage view, and
knowing which view answers which question is most of what this domain tests.

## What you will be able to do

- Name the billing unit for each Cortex service, and say why a resource monitor does not cap AI spend
- Pick the right `ACCOUNT_USAGE` view for a given cost question, at the right grain
- Attribute AI credits to a team using `ROLE_NAMES` and `QUERY_TAG`
- Cut token spend on an existing pipeline without rewriting it
- Recognise the costs that accrue when nobody is using the feature at all

## Before you start

- `ACCOUNTADMIN`, or a role granted access to the `SNOWFLAKE` database's `ACCOUNT_USAGE` schema
- Notebooks 3.1 and 3.2, for the privileges these queries assume
- Some Cortex usage in the account — the views are empty otherwise

📖 **Snowflake documentation for this notebook**
- [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)
- [CORTEX_AISQL_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_aisql_usage_history)
- [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)
- [Cortex Search costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-costs)
- [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)
- [QUERY_ATTRIBUTION_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history)


---
## One bill, several meters

Before any SQL: every Cortex service answers to a different meter. Getting this table straight is
most of the work.

| Service | Billing unit |
|---|---|
| Cortex AI functions over text | Tokens processed, priced per model |
| `AI_PARSE_DOCUMENT` | **Pages.** PDF and DOCX: one page bills as a page. Images (JPEG, JPG, TIF, TIFF, PNG): each file bills as a page. Text and HTML: each 3,000-character chunk bills as a page, the last chunk included |
| Cortex Search | Four separate streams — below |
| Cortex Analyst | **Messages** processed, not tokens. Tokens only enter the picture when Analyst is called through an agent. Warehouse credits for running the generated SQL are separate |
| Cortex Agents | Orchestration tokens, plus the cost of every tool the agent calls |
| Cortex AI Guardrails | Tokens scanned |
| Provisioned Throughput | Credits per PTU per hour across a one-month term, **charged for every allocated PTU whether or not you use it** |
| Snowpark Container Services | Compute-pool node-hours |

A **token** is the unit a model chews text into — roughly a short word. Input and output tokens are
both billed, which is why capping `max_tokens` matters as much as trimming the prompt.

### Cortex Search: four cost streams

1. **Virtual warehouse compute** — running the source query when the service is initialised and
   refreshed, including orchestrating the embedding jobs.
2. **`EMBED_TEXT` tokens** — charged per token embedded, each time rows are inserted or updated.
3. **Serving compute** — charged **per GB per month of uncompressed indexed data**, and you incur it
   while the service is available to answer queries, *even if no queries are served*.
4. **Storage**, at a flat rate per terabyte, plus **cloud services** compute for change detection —
   the latter billed only when the daily cloud-services cost exceeds 10% of the daily warehouse cost.

> The practical consequence: `TARGET_LAG` drives refresh and embedding cost; **index size** drives
> serving cost. A service nobody queries keeps billing until you suspend serving or drop it.

→ [More on Cortex Search costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-costs)
→ [More on AI_PARSE_DOCUMENT billing](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)


---
## Which view answers which question

All of these live in `SNOWFLAKE.ACCOUNT_USAGE`.

| View | Reports | Notes |
|---|---|---|
| `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` | Credits per AI function call, **including `AI_PARSE_DOCUMENT`** | `START_TIME, END_TIME, FUNCTION_NAME, MODEL_NAME, QUERY_ID, WAREHOUSE_ID, ROLE_NAMES, QUERY_TAG, USER_ID, METRICS, CREDITS, IS_COMPLETED`. Updated every 2 minutes best-effort, 5-minute SLA. Covers usage from 5 January 2026 |
| `CORTEX_AISQL_USAGE_HISTORY` | Token-level detail for SQL-invoked AI functions | `USAGE_TIME, MODEL_NAME, FUNCTION_NAME, TOKEN_CREDITS, TOKENS, TOKEN_CREDITS_GRANULAR, TOKENS_GRANULAR, QUERY_ID, QUERY_TAG, USER_ID, WAREHOUSE_ID`. Keyed on the hour the query completed. **Excludes `AI_PARSE_DOCUMENT`.** Covers usage from 17 November 2025 |
| `CORTEX_ANALYST_USAGE_HISTORY` | Analyst messages and credits | `START_TIME, END_TIME, REQUEST_COUNT, CREDITS, USERNAME` — a billing view only |
| `CORTEX_SEARCH_DAILY_USAGE_HISTORY` | Search credits per day, split by cost type | `USAGE_DATE, DATABASE_NAME, SCHEMA_NAME, SERVICE_NAME, SERVICE_ID, CONSUMPTION_TYPE, CREDITS, MODEL_NAME, TOKENS`. `CONSUMPTION_TYPE` is `SERVING`, `EMBED_TEXT_TOKENS` or `BATCH` |
| `CORTEX_SEARCH_SERVING_USAGE_HISTORY` | Search serving credits, hourly | `START_TIME, END_TIME, DATABASE_NAME, SCHEMA_NAME, SERVICE_NAME, SERVICE_ID, CREDITS` |
| `CORTEX_REST_API_USAGE_HISTORY` | Tokens per REST request | `START_TIME, END_TIME, REQUEST_ID, MODEL_NAME, TOKENS, TOKENS_GRANULAR, USER_ID, INFERENCE_REGION` |
| `CORTEX_AGENT_USAGE_HISTORY` | Tokens and credits per agent request | includes `AGENT_NAME`, `USER_NAME`, `TOKEN_CREDITS`, `TOKENS_GRANULAR`, `CREDITS_GRANULAR`, `METADATA` |
| `CORTEX_AI_GUARDRAILS_USAGE_HISTORY` | Guardrail scan tokens and credits | includes `GUARDRAILS_SIGNAL`, `GUARDRAIL_RESULTS`, `TOKEN_CREDITS`, `TOKENS` |
| `CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY` | PTU-hours | `PROVISIONED_THROUGHPUT_ID, INTERVAL_START_TIME, INTERVAL_END_TIME, CLOUD_SERVICE_PROVIDER, MODEL_NAME, TERM_START_DATE, TERM_END_DATE, PTU_COUNT, PTU_CREDITS` |
| `METERING_DAILY_HISTORY` | Credits per day per service type | `SERVICE_TYPE, USAGE_DATE, CREDITS_USED_COMPUTE, CREDITS_USED_CLOUD_SERVICES, CREDITS_USED, CREDITS_ADJUSTMENT_CLOUD_SERVICES, CREDITS_BILLED`. Latency up to 180 minutes |
| `METERING_HISTORY` | The same credits, hourly | has `SERVICE_TYPE`, `NAME`, `CREDITS_USED_COMPUTE/CLOUD_SERVICES/USED` — but **no** `CREDITS_BILLED` |
| `SNOWPARK_CONTAINER_SERVICES_HISTORY` | Hourly compute-pool credits | `START_TIME, END_TIME, COMPUTE_POOL_NAME, COMPUTE_POOL_ID, IS_EXCLUSIVE, APPLICATION_NAME, APPLICATION_ID, CREDITS_USED` |

### `SERVICE_TYPE = 'AI_SERVICES'`

`METERING_DAILY_HISTORY` and `METERING_HISTORY` both carry `SERVICE_TYPE`, and the value that covers
**Cortex AI Functions and Cortex Analyst** is `'AI_SERVICES'`. This is the highest-yield single fact
in 3.3.

> **The four-way distractor, and why each wrong answer is wrong:**
>
> | Candidate | Verdict |
> |---|---|
> | `ACCOUNT_USAGE.QUERY_HISTORY` | no credit column and no `SERVICE_TYPE` |
> | `INFORMATION_SCHEMA.METERING_HISTORY` | wrong schema — metering lives in `ACCOUNT_USAGE` |
> | `ACCOUNT_USAGE.METERING_HISTORY` | right family, but **hourly** |
> | `ACCOUNT_USAGE.METERING_DAILY_HISTORY` | **correct when the question says "daily"** |
>
> Read for the word *daily*. `METERING_DAILY_HISTORY` has up to 180 minutes of latency and 365 days of
> retention.

### Across accounts

`ORGANIZATION_USAGE` carries an org-wide counterpart for several of these —
`CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, `CORTEX_AGENT_USAGE_HISTORY`,
`CORTEX_SEARCH_SERVING_USAGE_HISTORY`, `METERING_HISTORY`, `METERING_DAILY_HISTORY` — plus
`USAGE_IN_CURRENCY_DAILY`, which converts usage to money. Reach for those when the question spans
more than one account.

→ [More on METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)
→ [More on the ORGANIZATION_USAGE schema](https://docs.snowflake.com/en/sql-reference/organization-usage)


In [ ]:
%%sql -r ai_usage_1
-- ============================================================
-- 1. CORTEX_AI_FUNCTIONS_USAGE_HISTORY — the comprehensive view
-- Columns: START_TIME, END_TIME, FUNCTION_NAME, MODEL_NAME, QUERY_ID, WAREHOUSE_ID,
--          ROLE_NAMES, QUERY_TAG, USER_ID, METRICS, CREDITS, IS_COMPLETED
-- ============================================================
SELECT
    DATE_TRUNC('day', START_TIME) AS day,
    FUNCTION_NAME,
    MODEL_NAME,
    COUNT(DISTINCT QUERY_ID)      AS call_count,
    SUM(CREDITS)                  AS total_credits,
    ROUND(SUM(CREDITS) / NULLIF(COUNT(DISTINCT QUERY_ID), 0), 4) AS avg_credits_per_query
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY total_credits DESC;


In [ ]:
%%sql -r ai_usage_2
-- Chargeback by role or by query tag (tag your pipelines!)
SELECT
    r.value::VARCHAR AS role_name,
    SUM(CREDITS)     AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY,
     LATERAL FLATTEN(ROLE_NAMES) r
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1 ORDER BY credits DESC;


> ### ⚠️ Common misconceptions
>
> **"I put a resource monitor on the warehouse, so AI spend is capped."**
> A resource monitor caps **warehouse** credits. Cortex serverless inference is metered separately and
> a runaway `AI_COMPLETE` batch will sail past a suspended warehouse trigger. The bill arrives anyway;
> the monitor never fires.
> → [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)
>
> **"`QUERY_ATTRIBUTION_HISTORY` will tell me which query burned the AI credits."**
> It attributes warehouse compute only. The documentation lists "costs for tokens processed by AI
> services" among the things it explicitly excludes. Use `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` with
> `QUERY_ID`, `ROLE_NAMES` and `QUERY_TAG` for AI attribution.
> → [QUERY_ATTRIBUTION_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history)
>
> **"Nobody queries that search service, so it costs nothing."**
> Serving is charged per GB per month of uncompressed indexed data while the service is available to
> answer queries, whether or not any arrive. A large, idle index is the quietest line on a Cortex bill.
> → [Cortex Search costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-costs)
>
> **"`CORTEX_AISQL_USAGE_HISTORY` covers every AI function."**
> It excludes `AI_PARSE_DOCUMENT`, which is billed per page rather than per token. A document pipeline
> audited only through that view looks free.
> → [CORTEX_AISQL_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_aisql_usage_history)


In [ ]:
%%sql -r ai_usage_3
-- ============================================================
-- 2. CORTEX_AISQL_USAGE_HISTORY — token-level detail, excludes AI_PARSE_DOCUMENT
-- Columns: USAGE_TIME, MODEL_NAME, FUNCTION_NAME, TOKEN_CREDITS, TOKENS,
--          TOKEN_CREDITS_GRANULAR, TOKENS_GRANULAR, QUERY_ID, QUERY_TAG, USER_ID, WAREHOUSE_ID
-- ============================================================
SELECT
    DATE_TRUNC('day', USAGE_TIME) AS day,
    FUNCTION_NAME,
    MODEL_NAME,
    SUM(TOKENS)        AS total_tokens,
    SUM(TOKEN_CREDITS) AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY total_credits DESC;


In [ ]:
%%sql -r ai_usage_4
-- TOKENS_GRANULAR / TOKEN_CREDITS_GRANULAR are OBJECTs — inspect one row before assuming key names.

-- 3. Cortex Analyst — billed PER MESSAGE
SELECT START_TIME, END_TIME, REQUEST_COUNT, CREDITS, USERNAME
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY START_TIME DESC;


In [ ]:
%%sql -r search_costs_1
-- ============================================================
-- Cortex Search costs, split by what you are paying for
-- CORTEX_SEARCH_DAILY_USAGE_HISTORY:
--   USAGE_DATE, DATABASE_NAME, SCHEMA_NAME, SERVICE_NAME, SERVICE_ID,
--   CONSUMPTION_TYPE ('SERVING' | 'EMBED_TEXT_TOKENS' | 'BATCH'),
--   CREDITS, MODEL_NAME, TOKENS
-- ============================================================
SELECT
    USAGE_DATE::DATE AS day,
    DATABASE_NAME || '.' || SCHEMA_NAME || '.' || SERVICE_NAME AS service,
    CONSUMPTION_TYPE,
    MODEL_NAME,
    SUM(TOKENS)  AS tokens,
    SUM(CREDITS) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE)
GROUP BY 1, 2, 3, 4
ORDER BY day DESC, credits DESC;


In [ ]:
%%sql -r search_costs_2
-- Pivot: indexing vs serving per service
SELECT
    DATABASE_NAME || '.' || SCHEMA_NAME || '.' || SERVICE_NAME AS service,
    SUM(IFF(CONSUMPTION_TYPE = 'EMBED_TEXT_TOKENS', CREDITS, 0)) AS indexing_credits,
    SUM(IFF(CONSUMPTION_TYPE = 'SERVING',           CREDITS, 0)) AS serving_credits,
    SUM(CREDITS)                                                 AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE)
GROUP BY 1 ORDER BY total_credits DESC;


In [ ]:
%%sql -r search_costs_3
-- Hourly serving detail.
-- Columns: START_TIME, END_TIME, DATABASE_NAME, SCHEMA_NAME, SERVICE_NAME, SERVICE_ID, CREDITS
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_SERVING_USAGE_HISTORY
ORDER BY 1 DESC LIMIT 20;

-- Levers:
--   * longer TARGET_LAG          -> fewer refreshes -> less warehouse + EMBED_TEXT cost
--   * fewer / narrower columns   -> smaller index   -> lower SERVING cost (per GB per month)
--   * AUTO_SUSPEND               -> serving suspends after inactivity, resumes on the next query
-- ALTER CORTEX SEARCH SERVICE ... SET TARGET_LAG = '24 hours';


In [ ]:
%%sql -r spcs_costs
-- Compute-pool credits. SNOWPARK_CONTAINER_SERVICES_HISTORY is the dedicated view;
-- METERING_HISTORY with SERVICE_TYPE = 'SNOWPARK_CONTAINER_SERVICES' is the metering-family answer.
SELECT
    START_TIME::DATE   AS day,
    COMPUTE_POOL_NAME,
    IS_EXCLUSIVE,
    SUM(CREDITS_USED)  AS credits_used
FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWPARK_CONTAINER_SERVICES_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_DATE)
GROUP BY 1, 2, 3
ORDER BY credits_used DESC;

-- A compute pool bills in the IDLE, ACTIVE, STOPPING and RESIZING states, and does NOT bill
-- while STARTING or SUSPENDED. Suspending an idle pool is therefore a real saving:
-- ALTER COMPUTE POOL GENAI_STUDY_CPU_POOL SUSPEND;
-- ALTER COMPUTE POOL GENAI_STUDY_CPU_POOL RESUME;


---
## Cutting token spend without rewriting the pipeline

In rough order of payoff:

1. **Use the smallest model that clears your accuracy bar.** The difference between model tiers is
   multiples per token, not percentages. Benchmark on a labelled sample before assuming you need the
   large one.
2. **Filter rows before the AI call**, and truncate boilerplate — signatures, quoted replies, legal
   footers — out of the text you send.
3. **Gate with `AI_COUNT_TOKENS`** and skip outlier rows. The documented signatures include
   `AI_COUNT_TOKENS('<function_name>', '<input_text>')` and
   `AI_COUNT_TOKENS('<function_name>', '<model_name>', '<input_text>')`, with function-specific forms
   for `ai_classify`, `ai_translate` and `ai_similarity`. It returns estimated **input** tokens.
4. **Cache.** Add a result column and process only rows where it is still `NULL`, or drive the job
   from a stream so each run touches new rows only.
5. **Prefer `AI_CLASSIFY` or `AI_EXTRACT`** over free-form `AI_COMPLETE` for structured tasks — fewer
   output tokens, and a parseable result.
6. **Cap `max_tokens`.** The default is 4096 and most tasks need a fraction of that.
7. **Use `page_filter` in `AI_PARSE_DOCUMENT`** so you pay for the pages you need. Snowflake also
   recommends running `AI_PARSE_DOCUMENT` on a warehouse no larger than `MEDIUM` — larger warehouses do
   not make it faster.

**The trade-off nobody mentions.** Every item on this list trades accuracy, latency or freshness for
credits. A smaller model misses edge cases. Truncation removes context. Caching means yesterday's
answer stands until something invalidates it. Decide which of those you can afford before you decide
how much to save.

→ [More on AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)


In [ ]:
%%sql -r token_optimization_1
-- Never call an AI function on a row you do not need.
-- AI_COUNT_TOKENS signatures include ('<function_name>', '<input_text>') and
-- ('<function_name>', '<model_name>', '<input_text>'); it returns estimated INPUT tokens.

-- Anti-pattern: AI on every row, every run
-- SELECT ticket_id, AI_COMPLETE('llama3.3-70b', ticket_text) FROM SUPPORT_TICKETS;

-- Better: filter -> truncate -> gate
SELECT
    ticket_id,
    AI_COMPLETE(
        'llama3.1-8b',
        'Classify ticket as billing, technical or shipping: ' || LEFT(ticket_text, 500)
    ) AS predicted_category
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE status = 'open'                                                     -- fewer rows
  AND category IS NULL                                                    -- not already done (cache)
  AND AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b', ticket_text) < 2000;  -- skip outliers


In [ ]:
%%sql -r token_optimization_2
-- Estimate before you run a batch
SELECT
    COUNT(*) AS rows_to_process,
    SUM(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b', LEFT(ticket_text, 500))) AS estimated_input_tokens
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE status = 'open';

-- Structured output is cheaper than free text for structured tasks, and a shallow response_format
-- schema is cheaper than a deep one.

-- Document parsing is billed per PAGE, so restrict pages rather than parsing whole documents:
-- SELECT AI_PARSE_DOCUMENT(TO_FILE('@DOCS_STAGE','contract.pdf'),
--                          {'mode':'OCR', 'page_filter':[{'start':0,'end':2}]});


> ### 🤔 Stop and think
>
> - A `TARGET_LAG` of one hour keeps your search index fresh and pays embedding costs every hour. A
>   lag of 24 hours pays once a day and can serve stale answers. Which of your indexed columns
>   genuinely changes hourly — and how would you find out rather than guess?
> - Provisioned Throughput bills for every allocated PTU for a whole month regardless of use. What
>   would you need to know about your traffic before committing to it, and what would you do in the
>   months when the shape of that traffic changes?
> - You can attribute AI spend by role or by query tag. Tags require every pipeline author to set one
>   consistently, forever. Which mechanism will still be accurate in a year, and what would you put in
>   place to keep it that way?


In [ ]:
%%sql
-- Object tagging for cost chargeback and monitoring
-- Tag AI-related objects so cost can be attributed to teams/projects

-- Create a cost center tag
CREATE TAG IF NOT EXISTS GENAI_STUDY.PUBLIC.COST_CENTER
    ALLOWED_VALUES 'ai_platform', 'data_science', 'bi_analytics', 'compliance';

-- Tag the Cortex Search service
ALTER CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH
    SET TAG GENAI_STUDY.PUBLIC.COST_CENTER = 'ai_platform';

-- Tag the SUPPORT_TICKETS table
ALTER TABLE GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    SET TAG GENAI_STUDY.PUBLIC.COST_CENTER = 'ai_platform';


In [ ]:
%%sql -r cost_tagging_2
-- What is tagged, and with what?
-- TAG_REFERENCES describes OBJECTS. It carries no credits, and there is no documented join key
-- into the AI credit views — so object tags support inventory and policy, not chargeback.
SELECT TAG_NAME, TAG_VALUE, OBJECT_DATABASE, OBJECT_SCHEMA, OBJECT_NAME, DOMAIN
FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
WHERE TAG_NAME = 'COST_CENTER'
  AND OBJECT_DELETED IS NULL
ORDER BY OBJECT_NAME;

-- For AI chargeback use ROLE_NAMES or QUERY_TAG in CORTEX_AI_FUNCTIONS_USAGE_HISTORY
-- (see the next cell). For warehouse compute use QUERY_ATTRIBUTION_HISTORY, remembering that
-- it excludes costs for tokens processed by AI services.


---
## The controls that exist, and the one that does not

| Control | Caps | Does not cap |
|---|---|---|
| Resource monitor on a warehouse | warehouse credits | Cortex serverless inference credits |
| `ALTER WAREHOUSE … SET AUTO_SUSPEND` | idle warehouse time | anything token-metered |
| `STATEMENT_TIMEOUT_IN_SECONDS` | how long one runaway statement can run | the tokens it burned before the timeout |
| Agent `orchestration.budget` (`seconds`, `tokens`) | one agent run's orchestration | tokens spent inside the agent's tools |
| Agent resource budgets | spend against an agent object, with a threshold to act on | anything outside that agent |
| Per-user credit quotas for agents | a user's monthly or daily agent credits | non-agent AI function calls |
| Query-side discipline — filters, truncation, model choice, caching | the tokens you actually send | nothing; this is the real lever |

`CREATE PROVISIONED THROUGHPUT` is an account-level privilege held by `ACCOUNTADMIN` by default;
revoking it is how you stop anyone else committing you to a month of reserved capacity.

→ [More on Cortex Agents cost controls](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)


In [ ]:
%%sql
-- ============================================================
-- SPEND CONTROLS
-- ============================================================

-- 1. Resource monitor — caps WAREHOUSE credits (the compute running your AI queries).
--    Token credits for Cortex serverless inference are NOT warehouse credits, so a resource
--    monitor alone does not cap AI-function spend.
CREATE OR REPLACE RESOURCE MONITOR GENAI_COST_LIMIT
    WITH CREDIT_QUOTA = 100
         FREQUENCY = MONTHLY
         START_TIMESTAMP = IMMEDIATELY
    TRIGGERS
        ON 80  PERCENT DO NOTIFY
        ON 100 PERCENT DO SUSPEND
        ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE COMPUTE_WH SET RESOURCE_MONITOR = GENAI_COST_LIMIT;


In [ ]:
%%sql -r usage_quotas_2
-- Resource monitors are read with SHOW, not from an ACCOUNT_USAGE view.
SHOW RESOURCE MONITORS;


In [ ]:
%%sql
-- The SHOW output includes credit_quota, used_credits, remaining_credits, level and frequency.

-- 2. Cortex Agents: orchestration budgets (seconds and tokens), resource budgets on the agent
--    object, and per-user credit quotas are the documented ways to cap agent spend.
--    The orchestration token budget excludes tokens spent inside tools.

-- 3. Provisioned Throughput: credits per PTU per hour for a one-month term, charged for every
--    allocated PTU whether or not you use it. Creating one needs the account-level
--    CREATE PROVISIONED THROUGHPUT privilege, held by ACCOUNTADMIN by default.

-- 4. Warehouse-side hygiene for AI workloads
ALTER WAREHOUSE COMPUTE_WH SET
    AUTO_SUSPEND = 60
    STATEMENT_TIMEOUT_IN_SECONDS = 3600;
-- Snowflake recommends a warehouse no larger than MEDIUM for AI_PARSE_DOCUMENT workloads.


In [ ]:
%%sql -r metering_daily_ai
-- Daily AI credit consumption — the canonical answer to a "daily AI cost" question.
-- ACCOUNT_USAGE (not INFORMATION_SCHEMA), and the DAILY view (not METERING_HISTORY).
SELECT
    USAGE_DATE,
    SERVICE_TYPE,
    CREDITS_USED_COMPUTE,
    CREDITS_USED_CLOUD_SERVICES,
    CREDITS_USED,
    CREDITS_BILLED
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE)
ORDER BY USAGE_DATE DESC;

In [ ]:
%%sql -r metering_hourly_ai
-- Same family, hourly grain — use this to locate a spike inside a day.
-- METERING_HISTORY has no CREDITS_BILLED column; that one is on METERING_DAILY_HISTORY.
SELECT
    START_TIME,
    SERVICE_TYPE,
    NAME,
    CREDITS_USED
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
WHERE SERVICE_TYPE IN ('AI_SERVICES', 'SNOWPARK_CONTAINER_SERVICES')
  AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY CREDITS_USED DESC;


In [ ]:
%%sql -r rest_api_usage
-- REST API consumption: which caller, which model, which region actually served it
SELECT
    DATE_TRUNC('day', START_TIME) AS day,
    MODEL_NAME,
    INFERENCE_REGION,
    COUNT(DISTINCT REQUEST_ID)    AS requests,
    SUM(TOKENS)                   AS tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_REST_API_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY tokens DESC;

In [ ]:
%%sql -r ptu_usage
-- Provisioned Throughput: are the reserved PTUs earning their keep?
-- PTUs bill per hour for the whole term whether or not you send traffic.
SELECT
    PROVISIONED_THROUGHPUT_ID,
    MODEL_NAME,
    CLOUD_SERVICE_PROVIDER,
    TERM_START_DATE,
    TERM_END_DATE,
    PTU_COUNT,
    SUM(PTU_CREDITS) AS credits_billed
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY
WHERE INTERVAL_START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY credits_billed DESC;

---
## Worked scenario: the bill tripled

**The situation.** Nightly batch jobs run `AI_COMPLETE('llama3.3-70b', …)` over 500,000 rows. The
monthly Cortex bill has tripled. Name three optimisations that do not require replacing the pipeline.

**1. Right-size the model.** A large model costs multiples of a small one per token. Benchmark
`llama3.1-8b` on a labelled sample and escalate only the rows where the small model is unsure. If the
task is classification into fixed labels, `AI_CLASSIFY` is cheaper than any free-form completion —
fewer output tokens and a parseable result.

**2. Cut input tokens.** Truncate boilerplate with `LEFT(ticket_text, 300)`, strip signatures and
quoted replies, and gate with `AI_COUNT_TOKENS` to skip outliers. Cap output with `max_tokens`; the
default is 4096 and a category label needs a handful.

**3. Stop reprocessing history.** Add a result column and process only `WHERE ai_result IS NULL`, or
drive the job from a stream so each night touches new rows only.

```sql
SELECT AI_COMPLETE('llama3.1-8b', LEFT(ticket_text, 300))
FROM SUPPORT_TICKETS
WHERE ai_result IS NULL
  AND status = 'open';
```

**Then prove it.** Compare before and after in `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` grouped by
`MODEL_NAME` and `QUERY_TAG`, having set `ALTER SESSION SET QUERY_TAG = 'nightly_triage'` so the
attribution is unambiguous.

> **The trap:** a resource monitor caps warehouse credits. Cortex serverless inference is billed
> separately, so a resource monitor alone will not stop a runaway AI batch. You cap it in the query —
> filters, truncation, model choice, caching — and, for agents, with orchestration budgets, resource
> budgets and per-user quotas.


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What is `AI_PARSE_DOCUMENT`'s billing unit, and how is a plain text file counted?

<details><summary>Show answer</summary>

Pages. For PDF and DOCX each page bills as a page; for image files each file bills as a page; for text
and HTML each 3,000-character chunk bills as a page, including the last chunk. Tokens are the wrong
answer here and it is a deliberate distractor — every other text function is token-metered.

→ [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**2.** Which `SERVICE_TYPE` value in `METERING_DAILY_HISTORY` covers Cortex AI functions and Cortex
Analyst?

<details><summary>Show answer</summary>

`'AI_SERVICES'`. Snowpark Container Services bills under `'SNOWPARK_CONTAINER_SERVICES'` and warehouse
compute under `'WAREHOUSE_METERING'`, so a query filtering for the wrong one comes back empty rather
than wrong — which is easy to misread as "we spent nothing".

→ [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)

</details>

**3.** What are the three `CONSUMPTION_TYPE` values in `CORTEX_SEARCH_DAILY_USAGE_HISTORY`?

<details><summary>Show answer</summary>

`SERVING`, `EMBED_TEXT_TOKENS` and `BATCH`. Splitting on this column is how you tell an expensive index
(high `SERVING`) from an expensive refresh cadence (high `EMBED_TEXT_TOKENS`) — two different problems
with two different fixes.

→ [CORTEX_SEARCH_DAILY_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_search_daily_usage_history)

</details>

**4.** You want to know how many input versus output tokens a document pipeline consumed. You query
`CORTEX_AISQL_USAGE_HISTORY` and get nothing. Why?

<details><summary>Show answer</summary>

That view excludes `AI_PARSE_DOCUMENT`, which is page-metered rather than token-metered.
`CORTEX_AI_FUNCTIONS_USAGE_HISTORY` covers all AI functions including that one, and reports credits.
There is no input/output token split to find, because the function is not billed on tokens at all.

→ [CORTEX_AISQL_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_aisql_usage_history)

</details>

**5.** A team's chatbot bill is high but `CORTEX_ANALYST_USAGE_HISTORY` shows a modest
`REQUEST_COUNT`. Where else should you look?

<details><summary>Show answer</summary>

At the warehouse credits for running the SQL Analyst generated, which are billed separately, and — if
Analyst is reached through an agent — at `CORTEX_AGENT_USAGE_HISTORY`, since tokens enter Analyst's
cost only on that path. Analyst itself is billed per message, so a modest request count really does
mean a modest Analyst charge.

→ [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

</details>

**6.** A resource monitor with a 100-credit monthly quota is attached to the warehouse running a
nightly `AI_COMPLETE` job. The job overruns its budget by a wide margin and the monitor never fires.
What went wrong?

<details><summary>Show answer</summary>

Nothing went wrong with the monitor. It caps warehouse credits, and the AI spend here is serverless
inference metered separately. The warehouse credits for orchestrating the query were genuinely small.
Cap this in the query, or move the workload behind an agent with budgets and quotas.

→ [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)

</details>

**7.** A Cortex Search service has had zero queries for a month and still shows daily credits. Is
this a billing error?

<details><summary>Show answer</summary>

No. Serving is charged per GB per month of uncompressed indexed data while the service is available to
answer queries, regardless of query volume. Either set `AUTO_SUSPEND` so serving suspends after a
period of inactivity, or drop the service. Reducing `TARGET_LAG` would make this worse, not better —
that lever affects refresh cost, not serving.

→ [Cortex Search costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-costs)

</details>

**8.** You want per-team chargeback for AI functions. `QUERY_ATTRIBUTION_HISTORY` returns credits that
look far too low. Why, and what do you use instead?

<details><summary>Show answer</summary>

`QUERY_ATTRIBUTION_HISTORY` attributes warehouse compute and explicitly excludes costs for tokens
processed by AI services. Use `CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, flattening `ROLE_NAMES` or grouping
on `QUERY_TAG`. The low number is not a bug — it is the warehouse half of the cost, correctly reported.

→ [QUERY_ATTRIBUTION_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history)

</details>

**9.** A pipeline has been tagged with `ALTER SESSION SET QUERY_TAG = 'nightly_triage'`. Which two
views let you report on that tag?

<details><summary>Show answer</summary>

`CORTEX_AI_FUNCTIONS_USAGE_HISTORY` and `CORTEX_AISQL_USAGE_HISTORY` both carry `QUERY_TAG`. Object
tags set with `ALTER TABLE … SET TAG` are a different mechanism entirely: they describe objects and do
not appear in the credit views, so they cannot be joined to credits directly.

→ [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)

</details>

**10.** Your team proposes Provisioned Throughput to make latency predictable. What does it cost, and
what would you want to see before agreeing?

<details><summary>Show answer</summary>

Credits per PTU per hour across a one-month term, charged for every allocated PTU whether or not you
send traffic. Before agreeing, you would want a sustained tokens-per-minute profile showing the demand
is steady rather than spiky — a bursty workload pays for a month of idle PTUs. Check utilisation in
`CORTEX_PROVISIONED_THROUGHPUT_USAGE_HISTORY` once it is running.

→ [Provisioned Throughput](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput)

</details>

**11.** You can cut a classification job's spend by moving from a large model to a small one, or by
truncating input to 300 characters. Which do you try first, and why?

<details><summary>Show answer</summary>

Model choice, because the price difference per token between tiers is a multiple while truncation is a
percentage — and because truncation silently changes what the model sees, so a quality regression is
harder to attribute. Benchmark the small model on a labelled sample, keep the large one as an
escalation path for low-confidence rows, and only then trim the input. Both cost accuracy; the model
swap is the one you can measure cleanly.

→ [Snowflake Cortex AISQL](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

</details>

**12.** Connecting to governance: a runaway AI batch was launched by a role you thought had no AI
access. Which view names the role, and which control in 3.1 would have prevented it?

<details><summary>Show answer</summary>

`CORTEX_AI_FUNCTIONS_USAGE_HISTORY` carries `ROLE_NAMES` — flatten it to see which roles were active
for the call. The prevention is the least-privilege work from 3.1: `USE AI FUNCTIONS` and
`SNOWFLAKE.CORTEX_USER` are granted to `PUBLIC` by default, so until they are revoked from `PUBLIC`
every role has AI access whether you granted it or not.

→ [CORTEX_AI_FUNCTIONS_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)

</details>
